<a href="https://colab.research.google.com/github/Abhi-crypto-code/Automatic-Generation-of-Control-Structures/blob/main/ByteT5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================
# 1. Install dependencies
# =========================
!pip install transformers datasets accelerate evaluate -q



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00


In [2]:
!pip install --upgrade transformers datasets


In [3]:
!pip install scikit-learn


In [4]:
# !pip uninstall torch torchvision torchaudio -y


In [5]:
# !pip uninstall torch

In [6]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [7]:
import torch
print(torch.cuda.is_available())  # should be True
print(torch.cuda.get_device_name(0))  # should show your GPU name


True
Tesla T4


In [8]:
# =========================
# 2. Imports
# =========================
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


In [9]:
# =========================
# 3. Config
# =========================
MODEL_NAME = "google/byt5-small"

TRAIN_FILE = "train_data_1k.json"   # <-- put your actual path
VAL_FILE   = "eval_data_1k.json"

OUTPUT_DIR = "./byt5_finetuned"

MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 512
BATCH_SIZE = 4
LEARNING_RATE = 3e-4
NUM_EPOCHS = 3
SEED = 42


In [10]:
# =========================
# 4. Load Dataset
# (supports CSV or JSON with input_text / target_text columns)
# =========================
ext = os.path.splitext(TRAIN_FILE)[1].lower()
if ext == ".csv":
    raw_datasets = load_dataset("csv", data_files={"train": TRAIN_FILE, "validation": VAL_FILE})
else:
    raw_datasets = load_dataset("json", data_files={"train": TRAIN_FILE, "validation": VAL_FILE})

print(raw_datasets)


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['PFD', 'PID'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['PFD', 'PID'],
        num_rows: 1000
    })
})


In [11]:
# print(raw_datasets["train"][0])           # original
# print(tokenized_datasets["train"][0])     # tokenized


In [12]:
# =========================
# 5. Tokenizer & Preprocessing
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_fn(batch):
    model_inputs = tokenizer(
        batch["PFD"],              # <-- use PFD as input
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        batch["PID"],              # <-- use PID as target
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length"
    )
    # Replace pad token IDs in labels with -100
    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in seq]
        for seq in labels["input_ids"]
    ]
    return model_inputs

tokenized_datasets = raw_datasets.map(
    preprocess_fn,
    batched=True,
    remove_columns=raw_datasets["train"].column_names
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [13]:
!pip install --upgrade torch

In [14]:
# =========================
# 6. Model + Data Collator
# =========================
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [15]:
# =========================
# 7. Metrics (Exact Match here, you can add BLEU/ROUGE if needed)
# =========================
metric = evaluate.load("accuracy")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    acc = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"accuracy": acc["accuracy"]}


In [17]:
# =========================
# 8. TrainingArguments
# =========================
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    do_train=True,
    do_eval=True,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=0.01,
    num_train_epochs=NUM_EPOCHS,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    seed=SEED
)



In [18]:
# =========================
# 9. Trainer
# =========================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


/tmp/ipython-input-1567210714.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [19]:
# =========================
# 10. Train
# =========================
trainer.train()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 2022chb1037 (2022chb1037-iit-ropar-tif) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,0.166200


TrainOutput(global_step=750, training_loss=0.1108265380859375, metrics={'train_runtime': 413.0011, 'train_samples_per_second': 7.264, 'train_steps_per_second': 1.816, 'total_flos': 2756248731648000.0, 'train_loss': 0.1108265380859375, 'epoch': 3.0})

In [20]:
# =========================
# 11. Save model
# =========================
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Model saved to:", OUTPUT_DIR)


Model saved to: ./byt5_finetuned


In [30]:
# =========================
# 12. Quick Inference
# =========================
# sample = raw_datasets["validation"][0]["input_text"]
sample = "(raw)(pp)(v)(v)(mix)<&|(raw)(hex)(v)&|(mix)<&|(raw)(v)&|(pp)(v)(mix)<1(r)(v)(splt)[(prod)](v)1"
print("INPUT:", sample)

outputs = model.generate(
    **tokenizer(sample, return_tensors="pt").to(device),
    max_length=MAX_TARGET_LEN,
    num_beams=5,
    num_return_sequences=5
)

decoded = tokenizer.decode(outputs[2], skip_special_tokens=True)
print("OUTPUT:", decoded)


INPUT: (raw)(pp)(v)(v)(mix)<&|(raw)(hex)(v)&|(mix)<&|(raw)(v)&|(pp)(v)(mix)<1(r)(v)(splt)[(prod)](v)1
OUTPUT: 
